In [0]:
import io, glob, hashlib, zipfile, requests
from datetime import datetime, timezone

ANO = 2026
URL = f"https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/IPE/DADOS/ipe_cia_aberta_{ANO}.zip"
BASE = f"/Volumes/cvm/landing/indices/ipe/ano={ANO}"

def ingestao_existente(base, sha):
    """Procura uma ingestão anterior com o mesmo conteúdo."""
    for arquivo in glob.glob(f"{base}/ingestao=*/_sha256.txt"):
        with open(arquivo) as f:
            if f.read().strip() == sha:
                return arquivo.split("/")[-2]
    return None

r = requests.get(URL, timeout=120)
r.raise_for_status()
conteudo = io.BytesIO(r.content)
if not zipfile.is_zipfile(conteudo):
    raise ValueError("A resposta não é um ZIP válido:\n" + r.text[:300])

with zipfile.ZipFile(conteudo) as z:
    nomes = sorted(z.namelist())
    sha = hashlib.sha256(b"".join(z.read(n) for n in nomes)).hexdigest()

    existente = ingestao_existente(BASE, sha)
    if existente:
        DESTINO = f"{BASE}/{existente}"
        print(f"Conteúdo idêntico já ingerido em {existente}. Nada gravado.")
    else:
        ingestao = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        DESTINO = f"{BASE}/ingestao={ingestao}"
        z.extractall(DESTINO)
        with open(f"{DESTINO}/_sha256.txt", "w") as f:
            f.write(sha)
        print(f"Novo conteúdo gravado em {DESTINO}")

### Ler CSV

In [0]:
df = (spark.read
      .option("header", True)
      .option("sep", ";")
      .option("encoding", "ISO-8859-1")
      .csv(DESTINO))

df.printSchema()
print("Linhas:", df.count())
display(df.limit(20))

In [0]:
from pyspark.sql import functions as F

(df.groupBy("Categoria", "Tipo")
   .count()
   .orderBy(F.desc("count"))
   .show(40, truncate=False))

In [0]:
df.groupBy("Tipo_Apresentacao", "Versao").count().orderBy("Versao").show(truncate=False)

In [0]:
import requests

link = (df.filter(F.col("Categoria") == "Dados Econômico-Financeiros")
          .select("Link_Download").first()[0])
print(link)

r = requests.get(link, timeout=120)
print("Status:", r.status_code)
print("Tipo:", r.headers.get("Content-Type"))
print("Content-Disposition:", r.headers.get("Content-Disposition"))
print("Tamanho:", len(r.content), "bytes")
print("Primeiros bytes:", r.content[:8])

In [0]:
from pyspark.sql import functions as F

# Empresas da amostra, identificadas pelo Codigo_CVM (texto, como está no índice)
EMPRESAS_ALVO = {
    "9512":  "Petrobras",
    "4170":  "Vale",
    "19348": "Itaú Unibanco",
    "5410":  "WEG",
    "22470": "Magazine Luiza",
}

# Tipos de documento que interessam ao projeto
TIPOS_ALVO = [
    "Press-release",
    "Apresentações a analistas/agentes do mercado",
]

alvo = df.filter(
    F.col("Codigo_CVM").isin(list(EMPRESAS_ALVO)) &
    F.col("Tipo").isin(TIPOS_ALVO)
)
print("Documentos-alvo no índice:", alvo.count())

# Contagem por empresa, para conferir se bate com a tabela anterior (59 no total)
alvo.groupBy("Codigo_CVM", "Nome_Companhia", "Tipo").count() \
    .orderBy("Codigo_CVM", "Tipo").show(truncate=False)

# Grupos com mais de uma linha: são versões do mesmo documento ou documentos diferentes?
(alvo.groupBy("Codigo_CVM", "Tipo", "Data_Referencia", "Assunto")
     .agg(F.count("*").alias("linhas"),
          F.collect_list("Versao").alias("versoes"),
          F.collect_list("Data_Entrega").alias("entregas"))
     .filter("linhas > 1")
     .orderBy("Codigo_CVM", "Data_Referencia")
     .show(truncate=False))

In [0]:
# A) O que aconteceu com a versão 1 da Vale 4T25?
(df.filter((F.col("Codigo_CVM") == "4170") & (F.col("Data_Referencia") == "2025-12-31"))
   .select("Categoria", "Tipo", "Assunto", "Versao", "Data_Entrega", "Protocolo_Entrega")
   .orderBy("Versao")
   .show(truncate=False))

# B) Sem o Assunto, a chave confunde documentos diferentes?
(alvo.groupBy("Codigo_CVM", "Tipo", "Data_Referencia")
     .agg(F.count("*").alias("linhas"),
          F.collect_list("Versao").alias("versoes"),
          F.collect_list("Assunto").alias("assuntos"))
     .filter("linhas > 1")
     .orderBy("Codigo_CVM", "Data_Referencia")
     .show(truncate=False))